# Multi-turn LongFact / LongFact++ in the format of Obeso et al. (2025)

Follows the paper's repo and data (`obalcells/hallucination_probes`), extended to multiple turns:

* **Prompts and splits:** taken verbatim from the paper's own datasets `obalcells/longfact-annotations` and `obalcells/longfact-augmented-annotations`
  (first user message of each row, keeping its `subset`, train/validation/test `split` and `canary`). The paper's test prompts are shared across models.
* **Generation settings:** as in the paper's Appendix C: temperature **0.1**, up to **2,048** new tokens per answer.
* **Turns:** turn 1 is the paper's prompt. Each later user turn is a follow-up question written by one fixed model (`FOLLOWUP_WRITER`)
  about the target model's **own** previous answer. Every assistant turn is generated by the target model.
* **Output format:** the paper's format, **one row per assistant turn**: `conversation` holds every message up to and including that turn,
  so the paper's annotation pipeline (which labels the last message, using the second-to-last as the question) and probe training code run on it unchanged.
  Extra columns: `conversation_id`, `turn`, `split`.
* **Saving:** one public dataset per model, `<your-username>/longfact-multiturn-<model>`, split files `data/<split>-<chunk>.jsonl`,
  uploaded chunk by chunk. If Kaggle stops, Run all again: finished chunks are skipped.
* **CuratorKIT pass (last cell):** length checks, exact + near-duplicate removal, secrets and toxicity screening on every turn. The raw `data/`
  files are never changed; turns that pass are published as a separate `curated` config, with CuratorKIT's provenance files under `curatorkit/`.

**Kaggle setup (once):** Settings → Accelerator → **GPU T4 x2**, Settings → **Internet on**, Add-ons → Secrets → **`HF_TOKEN`** (write access). Then Run all.

In [ ]:
import sys, shutil
assert sys.version_info >= (3, 11), f"CuratorKIT needs Python 3.11+, this runtime has {sys.version}"
!pip install -q -U datasets huggingface_hub "curatorkit[all]"
!apt-get -qq update && apt-get -qq install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh
assert shutil.which("ollama"), "Ollama did not install - read the output above (is Settings > Internet on?)"
print("Ollama installed at", shutil.which("ollama"))

In [ ]:
# ---------------- settings ----------------
# Ollama tag -> the Hugging Face model it corresponds to. q8_0 = 8-bit weights, closest to the paper's full-precision runs that fits a T4.
# Llama-3.1-8B and Qwen2.5-7B are the paper's own models (it released probes for both); Qwen3 would be "qwen3:8b-q8_0".
MODELS = {
    #"llama3.1:8b-instruct-q8_0": "meta-llama/Llama-3.1-8B-Instruct",
    #"qwen2.5:7b-instruct-q8_0":  "Qwen/Qwen2.5-7B-Instruct",
    "aya-expanse:8b-q8_0":       "CohereLabs/aya-expanse-8b",
}
FOLLOWUP_WRITER = "aya-expanse:8b"     # one fixed model writes every follow-up question (4-bit, so it fits next to the target)
PAPER_CONFIG = "Meta-Llama-3.1-8B-Instruct"   # whose split assignment to copy; the test prompts are the same for every model
SPLITS = ["test"]        # ["train", "validation", "test"] for everything
N_PER_SOURCE = 200       # prompts per split from each of LongFact and LongFact++; None = all of them
NUM_TURNS = 3            # assistant turns per conversation
TEMPERATURE = 0.1        # paper, Appendix C
MAX_NEW_TOKENS = 2048    # paper, Appendix C
NUM_CTX = 8192           # context window: must hold the whole conversation
CHUNK = 10               # conversations per uploaded file (checkpoint size)
WORKERS = 4              # conversations generated in parallel
REPO_PREFIX = "longfact-multiturn"
SEED = 0
# The paper appends this to LongFact questions; we append it to every follow-up so later turns stay just as entity-dense.
POSTAMBLE = "Provide as many specific details and examples as possible (such as names of people, numbers, events, locations, dates, times, etc.)"
FOLLOWUP_INSTRUCTIONS = (
    "You are writing the next message from a curious user chatting with an AI assistant. Read the conversation and write ONE natural "
    "follow-up question that asks for more specific factual detail (people, dates, numbers, places, organisations, works, citations) "
    "about something the assistant said in its last answer. Write only the question, nothing else.")

In [ ]:
import json, os, random, re, subprocess, time, urllib.request
from concurrent.futures import ThreadPoolExecutor
from datasets import load_dataset
from huggingface_hub import HfApi
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ["HF_TOKEN"]
api = HfApi(token=HF_TOKEN)
USER = api.whoami()["name"]

def log(msg):
    print(time.strftime("%H:%M:%S"), msg, flush=True)

# ---- prompts and splits, verbatim from the paper's datasets ----
prompts = []
for source in ["obalcells/longfact-annotations", "obalcells/longfact-augmented-annotations"]:
    for split in SPLITS:
        ds = load_dataset(source, PAPER_CONFIG, split=split, token=HF_TOKEN)
        rows = [{"subset": r["subset"], "canary": r["canary"], "split": split,
                 "prompt": next(m["content"] for m in r["conversation"] if m["role"] == "user")} for r in ds]
        random.Random(SEED).shuffle(rows)
        rows = rows[:N_PER_SOURCE] if N_PER_SOURCE else rows
        prompts += rows
        log(f"{source} [{split}]: using {len(rows)} of {len(ds)} prompts")
for i, p in enumerate(prompts):
    p["conversation_id"] = f"{p['split']}-{i:05d}"

# ---- Ollama server ----
server = subprocess.Popen([shutil.which("ollama"), "serve"],
                          env=dict(os.environ, OLLAMA_NUM_PARALLEL=str(WORKERS), OLLAMA_MAX_LOADED_MODELS="2"),
                          stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434"); break
    except Exception:
        time.sleep(2)
log("Ollama server running")

def chat(model, messages, system=None, max_tokens=MAX_NEW_TOKENS):
    msgs = ([{"role": "system", "content": system}] if system else []) + messages
    body = {"model": model, "messages": msgs, "stream": False, "keep_alive": "30m",
            "options": {"temperature": TEMPERATURE, "num_predict": max_tokens, "num_ctx": NUM_CTX}}
    if "qwen3" in model:
        body["think"] = False
    req = urllib.request.Request("http://localhost:11434/api/chat", data=json.dumps(body).encode(),
                                 headers={"Content-Type": "application/json"})
    out = json.load(urllib.request.urlopen(req, timeout=1800))["message"]["content"]
    return re.sub(r"<think>.*?</think>", "", out, flags=re.S).strip()

def pull(model):
    subprocess.run([shutil.which("ollama"), "pull", model], check=True, capture_output=True)

In [ ]:
def make_conversation(model, p):
    """One multi-turn conversation; returns one paper-format row per assistant turn."""
    conv, rows = [{"role": "user", "content": p["prompt"]}], []
    for turn in range(1, NUM_TURNS + 1):
        conv.append({"role": "assistant", "content": chat(model, conv)})
        rows.append({"subset": p["subset"], "model": MODELS[model], "conversation": [dict(m) for m in conv],
                     "canary": p["canary"], "conversation_id": p["conversation_id"], "turn": turn, "split": p["split"],
                     "ollama_tag": model, "followup_writer": FOLLOWUP_WRITER})
        if turn < NUM_TURNS:
            transcript = "\n\n".join(f"{m['role'].upper()}: {m['content']}" for m in conv)
            q = chat(FOLLOWUP_WRITER, [{"role": "user", "content": transcript}], system=FOLLOWUP_INSTRUCTIONS, max_tokens=200)
            q = q.strip().strip('"').strip()
            conv.append({"role": "user", "content": f"{q} {POSTAMBLE}\n"})
    return rows

pull(FOLLOWUP_WRITER)
for model in MODELS:
    repo_id = f"{USER}/{REPO_PREFIX}-{re.sub(r'[^a-zA-Z0-9.]+', '-', model).lower()}"
    api.create_repo(repo_id, repo_type="dataset", private=False, exist_ok=True)
    done = set(api.list_repo_files(repo_id, repo_type="dataset"))
    log(f"===== {model} -> https://huggingface.co/datasets/{repo_id}")
    pull(model)
    t = time.time()
    log(f"test reply: {chat(model, [{'role': 'user', 'content': 'Say hello in one short sentence.'}], max_tokens=30)!r} ({time.time() - t:.0f}s)")
    print(subprocess.run([shutil.which("ollama"), "ps"], capture_output=True, text=True).stdout)   # PROCESSOR should say GPU

    for split in SPLITS:
        todo = [p for p in prompts if p["split"] == split]
        for c in range(0, len(todo), CHUNK):
            name = f"data/{split}-{c // CHUNK:05d}.jsonl"
            if name in done:
                continue
            t = time.time()
            with ThreadPoolExecutor(WORKERS) as pool:
                results = list(pool.map(lambda p: make_conversation(model, p), todo[c:c + CHUNK]))
            os.makedirs("work", exist_ok=True)
            local = f"work/{re.sub(r'[^a-zA-Z0-9.]+', '-', model)}-{split}-{c // CHUNK:05d}.jsonl"
            with open(local, "w", encoding="utf-8") as f:
                for rows in results:
                    for r in rows:
                        f.write(json.dumps(r, ensure_ascii=False) + "\n")
            api.upload_file(path_or_fileobj=local, path_in_repo=name, repo_id=repo_id, repo_type="dataset",
                            commit_message=f"add {name}")
            log(f"  {name}: {len(results)} conversations x {NUM_TURNS} turns in {time.time() - t:.0f}s "
                f"[{min(c + CHUNK, len(todo))}/{len(todo)} {split}]")
    subprocess.run([shutil.which("ollama"), "stop", model], capture_output=True)
    log(f"===== {model} finished")
log("All models done.")

In [ ]:
# one example conversation per model (the last turn's row holds the whole conversation)
import glob
for model in MODELS:
    files = sorted(glob.glob(f"work/{re.sub(r'[^a-zA-Z0-9.]+', '-', model)}-*.jsonl"))
    if files:
        rows = [json.loads(l) for l in open(files[0], encoding="utf-8")]
        last = max(rows, key=lambda r: (r["conversation_id"] == rows[0]["conversation_id"], r["turn"]))
        print(f"\n################ {model} | {last['subset']} | {last['conversation_id']}")
        for m in last["conversation"]:
            print(f"\n[{m['role']}] {m['content'][:1500]}")

In [ ]:
# ---------------- CuratorKIT pass: quality checks + provenance (raw data/ stays untouched) ----------------
# clean=False and no PII pseudonymization: both would change the text, which breaks entity spans and fact-checking.
import glob
from huggingface_hub import snapshot_download
from curatorkit import Curator, CuratorConfig

CURATE = dict(
    clean               = False,     # keep every character exactly as the model wrote it
    dedup               = "exact",   # identical question+answer pairs
    min_tokens          = 20,        # empty / near-empty answers
    max_tokens          = 4096,      # runaway answers
    diversity_threshold = 0.95,      # near-identical turns (embedding similarity)
    secrets_gate        = True,      # keys, tokens, private keys
    toxicity_gate       = True,      # Detoxify; use toxicity_detoxify_model="multilingual" for Hindi data
    export_formats      = ["alpaca"],
    write_hf_readme     = False,
)

def readme(splits):
    lines = ["---", "configs:"]
    for cfg, pattern in (("raw", "data/{s}-*.jsonl"), ("curated", "curated/{s}.jsonl")):
        lines += [f"- config_name: {cfg}", "  data_files:"]
        lines += [f"  - split: {s}\n    path: {pattern.format(s=s)}" for s in splits]
    lines += ["---", "", "Multi-turn LongFact / LongFact++ generations in the format of Obeso et al. (2025), one row per assistant turn.",
              "`raw` = everything generated; `curated` = turns that passed the CuratorKIT checks (see `curatorkit/`)."]
    return "\n".join(lines) + "\n"

def curate(model):
    tag = re.sub(r"[^a-zA-Z0-9.]+", "-", model).lower()
    repo_id = f"{USER}/{REPO_PREFIX}-{tag}"
    base = f"curation/{tag}"
    snapshot_download(repo_id, repo_type="dataset", allow_patterns=["data/*.jsonl"], token=HF_TOKEN, local_dir=f"{base}/raw")
    rows = [json.loads(l) for f in sorted(glob.glob(f"{base}/raw/data/*.jsonl")) for l in open(f, encoding="utf-8")]
    if not rows:
        log(f"{model}: nothing generated yet"); return
    with open(f"{base}/flat.jsonl", "w", encoding="utf-8") as f:   # one CuratorKIT sample per assistant turn
        for r in rows:
            f.write(json.dumps({"id": f"{r['conversation_id']}-t{r['turn']}",
                                "instruction": r["conversation"][-2]["content"],
                                "output": r["conversation"][-1]["content"]}, ensure_ascii=False) + "\n")
    out = f"{base}/upload/curatorkit"
    Curator(CuratorConfig(dataset=f"{base}/flat.jsonl", output_dir=out, **CURATE)).run().print_summary()

    kept = {(x["instruction"], x["output"]) for x in map(json.loads, open(f"{out}/sft_alpaca.jsonl", encoding="utf-8"))}
    os.makedirs(f"{base}/upload/curated", exist_ok=True)
    splits = sorted({r["split"] for r in rows})
    for s in splits:
        with open(f"{base}/upload/curated/{s}.jsonl", "w", encoding="utf-8") as f:
            for r in rows:
                if r["split"] == s and (r["conversation"][-2]["content"], r["conversation"][-1]["content"]) in kept:
                    f.write(json.dumps(r, ensure_ascii=False) + "\n")
    os.remove(f"{out}/sft_alpaca.jsonl")          # the curated/ files replace it (they keep the paper's format)
    open(f"{base}/upload/README.md", "w").write(readme(splits))
    n_kept = sum(1 for r in rows if (r["conversation"][-2]["content"], r["conversation"][-1]["content"]) in kept)
    api.upload_folder(folder_path=f"{base}/upload", repo_id=repo_id, repo_type="dataset",
                      commit_message="CuratorKIT pass: curated config + provenance")
    log(f"{model}: {n_kept}/{len(rows)} turns passed CuratorKIT -> curated config; reasons in curatorkit/rejected.jsonl")

for model in MODELS:
    try:
        curate(model)
    except Exception as e:
        log(f"{model}: CuratorKIT pass failed: {type(e).__name__}: {e}")